# 6. Join MSA approvals to EIA customers → outputs

Links each utility in `msa_summary.csv` to its EIA utility ID(s) through `data/msa_eia_crosswalk.csv` (hand-reviewed), then builds:

- `outputs/msa_utilities_all.csv`: **every** EIA utility × state with residential customers and the Tesla / Enphase / ConnectDER status (blank = not approved). Utilities with nothing approved are the advocacy targets.
- `outputs/msa_state_summary.csv`: per state and US, the share of residential customers living in a service area where at least one device (and each brand) is approved.

Crosswalk rules:
- One MSA utility can be several EIA utilities (e.g. NV Energy = Nevada Power + Sierra Pacific).
- Several MSA utilities can share one EIA utility (the four former FirstEnergy Pennsylvania companies); each gets `customer_share` of its customers.
- Only **Approved** counts as covered. Pilot, Pending and Case-by-case show in the brand columns but don't count.

In [ ]:
import pandas as pd

from common import (CROSSWALK_CSV, EIA_CUSTOMERS_CSV, MSA_SUMMARY_CSV, STATE_SUMMARY_CSV,
                    UTILITIES_ALL_CSV, get_logger)
from mappings import APPROVED, BRANDS, STATE_NAMES, STATE_STD

log = get_logger("join")
customers = pd.read_csv(EIA_CUSTOMERS_CSV)
summary = pd.read_csv(MSA_SUMMARY_CSV, keep_default_na=False)
summary["state"] = summary["state"].map(STATE_STD)
crosswalk = pd.read_csv(CROSSWALK_CSV, keep_default_na=False)

## Crosswalk checks

In [ ]:
matched = crosswalk[crosswalk["eia_utility_id"] != ""].astype({"eia_utility_id": int, "customer_share": float})

# MSA utilities with no crosswalk row are left out until one is added.
known = set(zip(crosswalk["state"], crosswalk["utility"]))
missing = [f"{s} {u}" for s, u in zip(summary["state"], summary["utility"]) if (s, u) not in known]
if missing:
    log.warning("MSA utilities missing from %s (add rows; left out of totals): %s",
                CROSSWALK_CSV.name, "; ".join(missing))

# Crosswalk rows for utilities no longer on any manufacturer list.
stale = [f"{s} {u}" for s, u in known - set(zip(summary["state"], summary["utility"]))]
if stale:
    log.warning("crosswalk rows not in msa_summary (no longer listed?): %s", "; ".join(sorted(stale)))

# EIA IDs that don't exist in this EIA year (e.g. after a merger).
eia_keys = set(zip(customers["state"], customers["utility_id"]))
not_in_eia = [f"{s} {i}" for s, i in zip(matched["state"], matched["eia_utility_id"]) if (s, i) not in eia_keys]
if not_in_eia:
    log.warning("crosswalk EIA ids not in EIA data: %s", "; ".join(not_in_eia))

# Shares of one EIA utility shouldn't add up to more than 1.
over = matched.groupby(["state", "eia_utility_id"])["customer_share"].sum().loc[lambda s: s > 1.001]
assert over.empty, f"customer_share adds up to more than 1: {over.to_dict()}"
print(f"{len(summary)} MSA utilities, {len(matched)} crosswalk rows to EIA; "
      f"missing {len(missing)}, stale {len(stale)}, EIA ids not found {len(not_in_eia)}")

## Every EIA utility with brand statuses → `msa_utilities_all.csv`

In [ ]:
linked = (matched[["state", "utility", "eia_utility_id", "customer_share"]]
          .merge(summary, on=["state", "utility"])
          .rename(columns={"utility": "msa_utility", "eia_utility_id": "utility_id"}))
rows = customers.merge(linked, on=["state", "utility_id"], how="left")
rows["customer_share"] = rows["customer_share"].fillna(1.0)

# Any part of a split EIA utility not assigned to an MSA utility stays as its
# own row with no statuses.
leftover = 1 - rows.groupby(["state", "utility_id"])["customer_share"].transform("sum")
extra = rows[(leftover > 0.001) & rows["msa_utility"].notna()].drop_duplicates(["state", "utility_id"]).copy()
extra["customer_share"] = leftover[extra.index]
extra[["msa_utility", *BRANDS]] = None
rows = pd.concat([rows, extra])

for col in ("residential_customers", "population"):
    rows[col] = (rows[col] * rows["customer_share"]).round().astype(int)
rows[["msa_utility", *BRANDS]] = rows[["msa_utility", *BRANDS]].fillna("")
rows["any_device_approved"] = rows[BRANDS].eq(APPROVED).any(axis=1).map({True: "Yes", False: ""})
rows.insert(0, "state_name", rows["state"].map(STATE_NAMES))

utilities = (rows.rename(columns={"state": "state_abbr", "utility_id": "eia_utility_id"})
             [["state_name", "state_abbr", "utility_name", "eia_utility_id", "msa_utility", "residential_customers",
               "population", "people_per_household", "residential_estimated", *BRANDS, "any_device_approved",
               "ownership", "eia_year"]]
             .sort_values(["state_name", "utility_name", "msa_utility"], key=lambda c: c.str.lower()))

# The export keeps the readable columns and reports population; the full table
# (residential customers, state code, EIA ID, MSA name, etc.) stays in
# `utilities` for analysis.
UTILITIES_EXPORT_COLUMNS = {
    "state_name": "State",
    "utility_name": "Utility",
    "ownership": "Ownership",
    "population": "Population",
    **{b: b for b in BRANDS},
    "any_device_approved": "Any device approved",
}
utilities[list(UTILITIES_EXPORT_COLUMNS)].rename(columns=UTILITIES_EXPORT_COLUMNS).to_csv(UTILITIES_ALL_CSV, index=False)
log.info("wrote %s: %d rows, %d with an approved device", UTILITIES_ALL_CSV.name, len(utilities),
         utilities["any_device_approved"].eq("Yes").sum())

## Share of households covered → `msa_state_summary.csv`

In [ ]:
u = utilities.copy()
covered = {"any_device": u["any_device_approved"].eq("Yes"), **{b: u[b].eq(APPROVED) for b in BRANDS}}
for key, mask in covered.items():
    u[f"{key}_customers"] = u["residential_customers"].where(mask, 0)
    u[f"{key}_population"] = u["population"].where(mask, 0)
# Utilities are counted once per EIA utility, even when split across MSA utilities.
u["approved_utility_id"] = u["eia_utility_id"].where(covered["any_device"])

by_state = u.groupby(["state_name", "state_abbr"], as_index=False).agg(
    residential_customers=("residential_customers", "sum"),
    population=("population", "sum"),
    **{f"{k}_population": (f"{k}_population", "sum") for k in covered},
    utilities=("eia_utility_id", "nunique"),
    utilities_with_approval=("approved_utility_id", "nunique"),
    **{f"{k}_customers": (f"{k}_customers", "sum") for k in covered})
us = by_state.drop(columns=["state_name", "state_abbr"]).sum().to_frame().T.assign(
    state_name="United States", state_abbr="US")
states = pd.concat([by_state, us], ignore_index=True).rename(
    columns={"any_device_customers": "customers_with_any_approval",
             "any_device_population": "population_with_any_approval"})

states["pct_customers_any_approval"] = (states["customers_with_any_approval"] / states["residential_customers"] * 100).round(1)
for b in BRANDS:
    states[f"pct_customers_{b}_approved"] = (states[f"{b}_customers"] / states["residential_customers"] * 100).round(1)
states["pct_population_any_approval"] = (states["population_with_any_approval"] / states["population"] * 100).round(1)
for b in BRANDS:
    states[f"pct_population_{b}_approved"] = (states[f"{b}_population"] / states["population"] * 100).round(1)
states = states[["state_name", "state_abbr", "population", "population_with_any_approval",
                 "pct_population_any_approval", *[f"pct_population_{b}_approved" for b in BRANDS],
                 "residential_customers", "customers_with_any_approval", "pct_customers_any_approval",
                 *[f"pct_customers_{b}_approved" for b in BRANDS], "utilities", "utilities_with_approval"]]

# Readable export, by population; `states` also keeps the household (residential
# customer) versions and state codes for analysis.
STATES_EXPORT_COLUMNS = {
    "state_name": "State",
    "population": "Population",
    "population_with_any_approval": "Population where any device is approved",
    "pct_population_any_approval": "% of population where any device is approved",
    **{f"pct_population_{b}_approved": f"% of population where {b} is approved" for b in BRANDS},
    "utilities": "Utilities",
    "utilities_with_approval": "Utilities where any device is approved",
}
states[list(STATES_EXPORT_COLUMNS)].rename(columns=STATES_EXPORT_COLUMNS).to_csv(STATE_SUMMARY_CSV, index=False)

total = states.iloc[-1]
log.info("wrote %s: %s of %s residential customers (%.1f%%) and %s of %s people (%.1f%%) in a service area "
         "with at least one approval", STATE_SUMMARY_CSV.name, f"{total.customers_with_any_approval:,}",
         f"{total.residential_customers:,}", total.pct_customers_any_approval,
         f"{total.population_with_any_approval:,}", f"{total.population:,}", total.pct_population_any_approval)
states.sort_values("pct_customers_any_approval", ascending=False)

## Sanity checks

In [ ]:
# Every approved US utility in msa_summary should reach the output.
approved_msa = summary[summary[BRANDS].eq(APPROVED).any(axis=1) & summary["state"].isin(customers["state"])]
reached = set(zip(utilities.loc[utilities["any_device_approved"] == "Yes", "state_abbr"],
                  utilities.loc[utilities["any_device_approved"] == "Yes", "msa_utility"]))
not_reached = [f"{s} {u}" for s, u in zip(approved_msa["state"], approved_msa["utility"]) if (s, u) not in reached]
assert not not_reached, f"approved MSA utilities missing from output: {not_reached}"

# Coverage can't exceed 100%, and a brand can't cover more than any device.
for unit, total in (("customers", "residential_customers"), ("population", "population")):
    assert (states[f"{unit}_with_any_approval"] <= states[total]).all()
    for b in BRANDS:
        assert (states[f"pct_{unit}_{b}_approved"] <= states[f"pct_{unit}_any_approval"]).all()

# Customer totals survive the join (splits only move customers between rows).
for col in ("residential_customers", "population"):
    assert abs(utilities[col].sum() - customers[col].sum()) <= 5, col

# Approved utilities with very few EIA residential customers: worth a look in
# case the crosswalk points at the wrong EIA utility.
small = utilities[(utilities["any_device_approved"] == "Yes") & (utilities["residential_customers"] < 1000)]
print(f"All checks passed. Approved utilities with <1,000 residential customers ({len(small)}):")
small[["state_abbr", "utility_name", "msa_utility", "residential_customers"]]